In [1]:
# if the current working directory is src, change it to the parent directory
import os
if os.getcwd().endswith('experimental'):
    os.chdir('../..')

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
from scipy.integrate import quad
from soil_diskin.continuum_models import PowerLawDisKin
from soil_diskin.constants import LAMBDA_14C
from tqdm import tqdm

### Read results of powerlaw and lognormal fits

In [ ]:
# load data
ln = pd.read_csv('results/03_calibrate_models/03b_lognormal_predictions_calcurve.csv')

# convert lognromal parameters to mean and std
ln['sigma'] = np.sqrt(np.log(ln['pred'] / ln['turnover']))
ln['mu'] = -np.log(np.sqrt(ln['turnover']**3 / ln['pred']))

from matplotlib.colors import LogNorm
from matplotlib.collections import LineCollection

fig, ax = plt.subplots(dpi=300)

# make the colorbar with log scale
scatter = ax.scatter(ln['mu'], ln['sigma'], c=ln['pred'], cmap='viridis', norm=LogNorm())
ax.set(xlabel='$\mu$', ylabel='$\sigma$', title='$\mu$ vs $\sigma$ for log-normal model')
plt.colorbar(scatter, label='A')

# direction of steepest increase of log10(A) in the (mu, sigma) plane
logA = np.log10(ln['pred'].values)
c = np.array([ln['mu'].mean(), ln['sigma'].mean()])          # centroid
Xc = np.column_stack([ln['mu'].values - c[0], ln['sigma'].values - c[1]])
grad, *_ = np.linalg.lstsq(Xc, logA - logA.mean(), rcond=None)  # gradient of log10(A)
gmag = np.linalg.norm(grad)
u = grad / gmag                                               # unit vector toward increasing A

# bar spanning the data extent along the gradient direction, centred on the cloud
proj = Xc @ u
p0, p1 = c + u * proj.min(), c + u * proj.max()
ts = np.linspace(0, 1, 100)
pts = p0[None, :] + (p1 - p0)[None, :] * ts[:, None]
segs = np.stack([pts[:-1], pts[1:]], axis=1)

# colour the bar with the same gradient/norm as the scatter (dark = low A, bright = high A)
A_line = 10 ** (logA.mean() + ((pts - c) @ u) * gmag)
bar = LineCollection(segs, cmap=scatter.cmap, norm=scatter.norm, linewidth=6,
                     capstyle='round', zorder=5)
bar.set_array(A_line[:-1])
ax.add_collection(bar)

# annotate the maximal variation in log(A)
log_var = logA.max() - logA.min()  # log10(A_max / A_min)
ax.annotate(f'max $\Delta\log_{{10}}$(A) = {log_var:.2f} dex', xy=p1,
            xytext=(5, 5), textcoords='offset points', va='bottom', ha='left')